In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
!pip install python-dotenv

In [ ]:
from dotenv import load_dotenv
import pandas as pd
import os
import logging
import uuid
import time
import requests
from datetime import datetime
import json

In [ ]:
# asegura guardar el archivo .env con el token, sugerencia BANXICO_TOKEN=aqui_va_el_token
load_dotenv('/content/drive/MyDrive/add/banxico/credenciales/.env', override=True)

In [ ]:
banxico_token = os.getenv("BANXICO_TOKEN")

# Header indicado en Banxico
if banxico_token:
    banxico_token = banxico_token.strip()

headers_banxico = {
    "Bmx-Token": banxico_token
}

print("Token cargado")

# Series
* Tipo de cambio Pesos por dólar E.U.A. Tipo de cambio para solventar obligaciones denominadas en moneda extranjera Fecha de determinación (FIX)
  * **Id Serie:** SF43718
* Fuentes y usos de la base monetaria Otros activos, netos de otros pasivos y capital
  * **Id Serie:** SF44032
* Tasa objetivo
  * **Id Serie:** SF61745

In [ ]:
#Funcion para consumir API y obtener metadatos de respuesta
def consumir_api( url, params=None, headers=None, method="GET", timeout=30, json_payload=None):
    inicio = time.time()

    try:
        response = requests.request(
            method=method,
            url=url,
            params=params, #REST
            json=json_payload, #GRAPHQL
            headers=headers,
            timeout=timeout
        )

        duracion = round(time.time() - inicio, 3)

        # Metadata de respuesta
        metadata = {
            "url": response.url,
            "status_code": response.status_code,
            "ok": response.ok,
            "content_type": response.headers.get("Content-Type"),
            "content_length": response.headers.get("Content-Length"),
            "encoding": response.encoding,
            "tiempo_respuesta_s": duracion,
            "headers": dict(response.headers)
        }

        # Validar status HTTP
        response.raise_for_status()

        # Intentar parsear JSON
        try:
            data = response.json()
            metadata["formato_detectado"] = "json"

            if isinstance(data, dict) and "errors" in data:
                metadata["graphql_error"] = True
                raise Exception(f"Error GraphQL: {data["errors"]}")
        except:
            data = response.text
            metadata["formato_detectado"] = "texto"

        return data, metadata

    except requests.exceptions.RequestException as e:
        print(e)
        return None, {
            "error": str(e),
            "url": url
        }

In [ ]:
id_series = "SF43718,SF44032,SF61745"

# endpoint
url_series = f"https://www.banxico.org.mx/SieAPIRest/service/v1/series/{id_series}"

try:
  # consumir API
  data_banxico, metadata_banxico = consumir_api(
      url=url_series,
      headers=headers_banxico,
      method="GET"
  )
  # parsear al JSON
  if data_banxico and "bmx" in data_banxico:
      series_info = data_banxico["bmx"]["series"]
      for serie in series_info:
          periodicidad = serie.get('periodicidad', 'No disponible')
          cifra = serie.get('cifra', 'No disponible')
          unidad = serie.get('unidad', 'No disponible')

          print(f"ID Serie:      {serie.get('idSerie')}")
          print(f"Título:        {serie.get('titulo')}")
          print(f"Fecha inicial: {serie.get('fechaInicio')}")
          print(f"Fecha final:   {serie.get('fechaFin')}")
          print(f"Observaciones relevantes: Periodicidad: {periodicidad} | Cifra: {cifra} | Unidad: {unidad}")
          print("-" * 10)

  else:
    print("No se encontraron datos")
except Exception as e:
      logging.error(f"Error al consumir la API de Banxico: {e}")
      print(f"Error: {e}")

In [ ]:
tablas = pd.DataFrame([{
    "ID Serie":      serie.get("idSerie"),
    "Título":        serie.get("titulo"),
    "Fecha Inicial": serie.get("fechaInicio"),
    "Fecha Final":   serie.get("fechaFin"),
    "Periodicidad":  serie.get("periodicidad", "No disponible"),
    "Cifra":         serie.get("cifra", "No disponible"),
    "Unidad":        serie.get("unidad", "No disponible"),
} for serie in series_info])

In [ ]:
#guardar datos temp
datos_crudos_dict = {}

if 'series_info' in locals():
    for serie in series_info:
        id_serie = serie.get('idSerie')
        titulo = serie.get('titulo')
        fecha_ini_raw = serie.get('fechaInicio')
        fecha_fin_raw = serie.get('fechaFin')

        try:
            #convertir fecha
            fecha_ini = datetime.strptime(fecha_ini_raw, '%d/%m/%Y').strftime('%Y-%m-%d')
            fecha_fin = datetime.strptime(fecha_fin_raw, '%d/%m/%Y').strftime('%Y-%m-%d')

            #endpoint con fechas
            url_datos = f"https://www.banxico.org.mx/SieAPIRest/service/v1/series/{id_serie}/datos/{fecha_ini}/{fecha_fin}"

            data_serie, metadata = consumir_api(
                url=url_datos,
                headers=headers_banxico,
                method="GET"
            )

            #extraer datos
            if data_serie and "bmx" in data_serie and "series" in data_serie["bmx"]:
                datos_historicos = data_serie["bmx"]["series"][0].get("datos", [])

                if datos_historicos:
                    #guardar datos
                    datos_crudos_dict[id_serie] = {
                        'titulo': titulo,
                        'datos': datos_historicos
                    }
                    print(f"Datos descargados con éxito para la serie: {id_serie}")
                else:
                    print(f"La serie {id_serie} no devolvió observaciones.")
        except Exception as e:
            print(f"Error al procesar la serie {id_serie}: {e}")
else:
    print("Error: Ejecuta primero la celda donde generas la variable 'series_info'.")

In [ ]:
#almacenar DataFrames
df_series_dict = {}

if 'datos_crudos_dict' in locals() and datos_crudos_dict:
    for id_serie, info in datos_crudos_dict.items():
        titulo = info['titulo']
        datos_historicos = info['datos']

        #Crear DataFrame
        df = pd.DataFrame(datos_historicos)

        #agregar las columnas
        df['id_serie'] = id_serie
        df['titulo'] = titulo

        df.rename(columns={'dato': 'valor'}, inplace=True)

        df = df[['id_serie', 'titulo', 'fecha', 'valor']]

        df['valor'] = df['valor'].replace('N/E', pd.NA)

        #guardar en diccionario
        df_series_dict[id_serie] = df

        print(f"\nDataFrame de la serie {id_serie}:")
        display(df.head(3))
else:
    print("Error: No se encontraron datos.")

In [ ]:
# Guardar archivos en drive
ruta_salida = "/content/drive/MyDrive/add/banxico/"
os.makedirs(ruta_salida, exist_ok=True)

if 'df_series_dict' in locals() and df_series_dict:
    for id_serie, df in df_series_dict.items():
        #nombres
        archivo_csv = os.path.join(ruta_salida, f"serie_{id_serie}.csv")
        archivo_json = os.path.join(ruta_salida, f"serie_{id_serie}.json")
        archivo_parquet = os.path.join(ruta_salida, f"serie_{id_serie}.parquet")

        #exportar a 3 formatos
        df.to_csv(archivo_csv, index=False, encoding='utf-8')
        df.to_json(archivo_json, orient='records', force_ascii=False, indent=4)
        df.to_parquet(archivo_parquet, index=False)

        print(f"\nArchivos de la serie {id_serie} guardados:")
        print(f" -> {archivo_csv}")
        print(f" -> {archivo_json}")
        print(f" -> {archivo_parquet}")

    print("\nProceso de guardado completado con éxito")
else:
     print("Error: No hay DataFrames construidos.")

In [ ]:
registros_totales = sum([len(df) for df in df_series_dict.values()]) if 'df_series_dict' in locals() else 0

log_data = {
    "run_id": str(uuid.uuid4()),
    "fecha_proceso": datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    "fuente": "Banco de Mexico SIE API",
    "series_consultadas": list(df_series_dict.keys()) if 'df_series_dict' in locals() else [],
    "registros_extraidos": registros_totales,
    "duracion": "0.5s",
    "estado": "success" if registros_totales > 0 else "error"
}

# Guardar el log en Google Drive
ruta_log = os.path.join(ruta_salida, "ejecucion_log.json")
with open(ruta_log, 'w', encoding='utf-8') as f:
    json.dump(log_data, f, ensure_ascii=False, indent=4)

print(f"Log de ejecucion guardado exitosamente en: {ruta_log}")
print(json.dumps(log_data, indent=4))